In [ ]:
import os
import subprocess
import sys

# Suppress outputs for pip install
subprocess.run(["pip", "install", "configilm"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["pip", "install", "lightning"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["pip", "install", "lmdb"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Suppress output for git clone
subprocess.run(["git", "clone", "https://git.tu-berlin.de/rsim/reben-training-scripts.git"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Change directory without output
# os.chdir('/content/reben-training-scripts')
# from reben_publication.BigEarthNetv2_0_ImageClassifier import BigEarthNetv2_0_ImageClassifier
sys.path.append('/content/reben-training-scripts')


In [ ]:
!pip uninstall -y tensorflow
!pip install tensorflow-cpu

Found existing installation: tensorflow 2.17.1
Uninstalling tensorflow-2.17.1:
  Successfully uninstalled tensorflow-2.17.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.0/230.0 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 113.8 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.17.1
    Uninstalling tensorboard-2.17.1:
      Successfully uninstalled tensorboard-2.17.1


In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.image import resize
from tqdm import tqdm

import json
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, random_split, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau, LambdaLR
from torch.cuda.amp import GradScaler, autocast
import torch.optim as optim
from torch.nn import BCEWithLogitsLoss
from sklearn.metrics import classification_report

from reben_publication.BigEarthNetv2_0_ImageClassifier import BigEarthNetv2_0_ImageClassifier
from configilm.extra.BENv2_utils import band_combi_to_mean_std, STANDARD_BANDS

### Initialize the Preprocessor

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
sys.path.append('/content/drive/My Drive/EuroSAT')


In [ ]:
from configilm.extra.BENv2_utils import band_combi_to_mean_std, STANDARD_BANDS
from preprocessing import EuroSATPreprocessor, TensorflowToTorchDataset

# Get mean and std for normalization
means, stds = band_combi_to_mean_std(STANDARD_BANDS[10], interpolation="120_bilinear")

# Initialize the preprocessor
preprocessor = EuroSATPreprocessor(input_size=120, band_indices=[1, 2, 3, 4, 5, 6, 7, 8, 10, 11], means=means, stds=stds)


In [ ]:
# Save the dataset to a specific directory
save_path = '/content/eurosat_dataset'
dataset, info = tfds.load("eurosat/all", split='train', with_info=True, data_dir=save_path)


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...:   0%|          | 0/27000 [00:00<?, ? examples/s]

Shuffling /content/eurosat_dataset/eurosat/all/incomplete.8LGXJ3_2.0.0/eurosat-train.tfrecord*...:   0%|      …

Dataset eurosat downloaded and prepared to /content/eurosat_dataset/eurosat/all/2.0.0. Subsequent calls will reuse this data.


In [ ]:
# Inspect dataset labels
for sample in dataset.take(5):  # Inspect 5 samples
    label = sample['label']
    print("Label Type:", type(label.numpy()))
    print("Label Shape:", label.shape)
    print("Label Value:", label.numpy())
    print("-" * 30)


Label Type: <class 'numpy.int64'>
Label Shape: ()
Label Value: 8
------------------------------
Label Type: <class 'numpy.int64'>
Label Shape: ()
Label Value: 4
------------------------------
Label Type: <class 'numpy.int64'>
Label Shape: ()
Label Value: 5
------------------------------
Label Type: <class 'numpy.int64'>
Label Shape: ()
Label Value: 5
------------------------------
Label Type: <class 'numpy.int64'>
Label Shape: ()
Label Value: 3
------------------------------


EuroSAT dataset is multi class

In [ ]:
# Apply preprocessing using map
def preprocess_sample(sample):
    """Preprocess a single sample using the EuroSATPreprocessor."""
    return preprocessor.preprocess_sample(sample)

# Preprocess the entire dataset
preprocessed_dataset = dataset.map(preprocess_sample, num_parallel_calls=tf.data.AUTOTUNE)

# Check a preprocessed sample
for sample in preprocessed_dataset.take(1):
    print("Filename:", sample['filename'].numpy())
    print("Image Shape:", sample['sentinel2'].shape)
    print("Label:", sample['label'].numpy())

Filename: b'River_15.tif'
Image Shape: (120, 120, 10)
Label: 8


In [ ]:
# Convert TensorFlow dataset to PyTorch dataset
torch_dataset = TensorflowToTorchDataset(preprocessed_dataset)

# Calculate dataset sizes for train, validation, and test splits
dataset_size = len(torch_dataset)
train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size  # Ensures no data is left out

# Set the random seed for reproducibility
torch.manual_seed(42)

# Split the dataset
train_dataset, val_dataset, test_dataset = random_split(torch_dataset, [train_size, val_size, test_size])

# Create DataLoaders for each split
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Check a batch from the train_loader
for images, labels in train_loader:
    print("Batch Image Shape:", images.shape)  # [batch_size, channels, height, width]
    print("Batch Label Shape:", labels.shape)  # [batch_size]
    break


Batch Image Shape: torch.Size([64, 10, 120, 120])
Batch Label Shape: torch.Size([64])


In [ ]:
for _, labels in train_loader:
    print("Label Shape:", labels.shape)
    print("Sample Label:", labels[0])
    break


Label Shape: torch.Size([64])
Sample Label: tensor(1)


In [ ]:
for images, labels in train_loader:
    print(f"Image Shape: {images.shape}")  # [batch_size, channels, height, width]
    print(f"Label Shape: {labels.shape}")  # [batch_size]
    print(f"Sample Labels: {labels[:5]}")
    break


Image Shape: torch.Size([64, 10, 120, 120])
Label Shape: torch.Size([64])
Sample Labels: tensor([7, 1, 8, 5, 8])


### Loading the pre trained model

In [ ]:
model = BigEarthNetv2_0_ImageClassifier.from_pretrained(
  "BIFOLD-BigEarthNetv2-0/vit_base_patch8_224-s2-v0.1.1")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

In [ ]:
model.config

ILMConfiguration(timm_model_name='vit_base_patch8_224', hf_model_name=None, image_size=120, channels=10, classes=19, class_names=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18'], network_type=0, visual_features_out=512, fusion_in=512, fusion_out=512, fusion_hidden=256, v_dropout_rate=0.25, t_dropout_rate=0.25, fusion_dropout_rate=0.25, _fusion_method='torch.mul', _fusion_activation='nn.Tanh()', drop_rate=0.15, drop_path_rate=0.15, use_pooler_output=True, max_sequence_length=32, load_pretrained_timm_if_available=False, load_pretrained_hf_if_available=True, custom_fusion_method=None, custom_fusion_activation=None)

In [ ]:
model

BigEarthNetv2_0_ImageClassifier(
  (model): ConfigILM(
    (vision_encoder): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(10, 768, kernel_size=(8, 8), stride=(8, 8))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): 

#### Freeze N Layers

In [ ]:
def unfreeze_last_n_blocks(model, num_unfreeze_blocks, num_classes=10):
    """
    Unfreeze the last N blocks of the Vision Transformer model and update the classification head.

    Args:
        model: The Vision Transformer model (BigEarthNetv2_0_ImageClassifier).
        num_unfreeze_blocks: Number of blocks to unfreeze.
        num_classes: Number of output classes for classification.
    """
    # Reference to the blocks in the Vision Transformer
    vit_blocks = model.model.vision_encoder.blocks
    total_blocks = len(vit_blocks)

    # Freeze all parameters by default
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze the last N blocks
    for i in range(total_blocks - num_unfreeze_blocks, total_blocks):
        for param in vit_blocks[i].parameters():
            param.requires_grad = True

    # Update and unfreeze the final classification head to have num_classes outputs
    in_features = model.model.vision_encoder.head.in_features
    new_head = nn.Linear(in_features, num_classes)

    # Initialize the new classification head
    nn.init.xavier_uniform_(new_head.weight)
    if new_head.bias is not None:
        nn.init.constant_(new_head.bias, 0)

    # Move the new head to the same device as the model's parameters
    device = next(model.parameters()).device
    new_head = new_head.to(device)

    # Replace the old head with the new one
    model.model.vision_encoder.head = new_head

    # Ensure the new head's parameters are trainable
    for param in model.model.vision_encoder.head.parameters():
        param.requires_grad = True

    print(f"Unfroze the last {num_unfreeze_blocks} blocks and updated the classification head to {num_classes} classes.")

    # Verification: Print trainable parameters and their devices
    # trainable_params = [name for name, param in model.named_parameters() if param.requires_grad]
    # print("Trainable Parameters and their devices:")
    # for name in trainable_params:
    #     param = dict(model.named_parameters())[name]
    #     print(f"{name}: {param.device}")

In [ ]:
# Number of blocks to unfreeze
num_unfreeze_blocks = 3

# Unfreeze the last N blocks
unfreeze_last_n_blocks(model, num_unfreeze_blocks)

# Check if parameters are properly frozen/unfrozen
for name, param in model.named_parameters():
    print(f"{name} - Requires Grad: {param.requires_grad}")


Unfroze the last 3 blocks and updated the classification head to 10 classes.
model.vision_encoder.cls_token - Requires Grad: False
model.vision_encoder.pos_embed - Requires Grad: False
model.vision_encoder.patch_embed.proj.weight - Requires Grad: False
model.vision_encoder.patch_embed.proj.bias - Requires Grad: False
model.vision_encoder.blocks.0.norm1.weight - Requires Grad: False
model.vision_encoder.blocks.0.norm1.bias - Requires Grad: False
model.vision_encoder.blocks.0.attn.qkv.weight - Requires Grad: False
model.vision_encoder.blocks.0.attn.qkv.bias - Requires Grad: False
model.vision_encoder.blocks.0.attn.proj.weight - Requires Grad: False
model.vision_encoder.blocks.0.attn.proj.bias - Requires Grad: False
model.vision_encoder.blocks.0.norm2.weight - Requires Grad: False
model.vision_encoder.blocks.0.norm2.bias - Requires Grad: False
model.vision_encoder.blocks.0.mlp.fc1.weight - Requires Grad: False
model.vision_encoder.blocks.0.mlp.fc1.bias - Requires Grad: False
model.vision_

#### Update the Final Layer

In [ ]:
# Update the final classification head for 10 classes
model.model.vision_encoder.head = nn.Linear(in_features=768, out_features=10)

# Check if the layer is updated
print(model.model.vision_encoder.head)

Linear(in_features=768, out_features=10, bias=True)


In [ ]:
model

BigEarthNetv2_0_ImageClassifier(
  (model): ConfigILM(
    (vision_encoder): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(10, 768, kernel_size=(8, 8), stride=(8, 8))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): 

#### Early Stopping

In [ ]:
# Verify that all trainable parameters are on the correct device
trainable_params = [name for name, param in model.named_parameters() if param.requires_grad]
print("Trainable Parameters and their devices:")
for name in trainable_params:
    param = dict(model.named_parameters())[name]
    print(f"{name}: {param.device}")


Trainable Parameters and their devices:
model.vision_encoder.blocks.9.norm1.weight: cpu
model.vision_encoder.blocks.9.norm1.bias: cpu
model.vision_encoder.blocks.9.attn.qkv.weight: cpu
model.vision_encoder.blocks.9.attn.qkv.bias: cpu
model.vision_encoder.blocks.9.attn.proj.weight: cpu
model.vision_encoder.blocks.9.attn.proj.bias: cpu
model.vision_encoder.blocks.9.norm2.weight: cpu
model.vision_encoder.blocks.9.norm2.bias: cpu
model.vision_encoder.blocks.9.mlp.fc1.weight: cpu
model.vision_encoder.blocks.9.mlp.fc1.bias: cpu
model.vision_encoder.blocks.9.mlp.fc2.weight: cpu
model.vision_encoder.blocks.9.mlp.fc2.bias: cpu
model.vision_encoder.blocks.10.norm1.weight: cpu
model.vision_encoder.blocks.10.norm1.bias: cpu
model.vision_encoder.blocks.10.attn.qkv.weight: cpu
model.vision_encoder.blocks.10.attn.qkv.bias: cpu
model.vision_encoder.blocks.10.attn.proj.weight: cpu
model.vision_encoder.blocks.10.attn.proj.bias: cpu
model.vision_encoder.blocks.10.norm2.weight: cpu
model.vision_encoder.bl

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, monitor="val_loss", mode="min"):
        """
        Initializes the EarlyStopping object.

        Args:
            patience (int): How long to wait after last time monitored metric improved.
            monitor (str): Which metric to monitor ('val_loss', 'val_accuracy', etc.).
            mode (str): 'min' to minimize the monitored metric, 'max' to maximize.
        """
        self.patience = patience
        self.monitor = monitor
        self.mode = mode
        self.counter = 0
        self.early_stop = False
        self.best_score = float('inf') if mode == 'min' else float('-inf')

    def __call__(self, current_value):
        if self.mode == "min":
            if current_value < self.best_score:
                self.best_score = current_value
                self.counter = 0
            else:
                self.counter += 1
        elif self.mode == "max":
            if current_value > self.best_score:
                self.best_score = current_value
                self.counter = 0
            else:
                self.counter += 1

        if self.counter >= self.patience:
            self.early_stop = True

In [ ]:
# Inspect the current classification head
print(model.model.vision_encoder.head)


Linear(in_features=768, out_features=10, bias=True)


In [ ]:
import torch
print(torch.__version__)


2.5.1+cu121


Defining a Lambda Function for Learning Rate Scheduling

In [ ]:
import math

def lr_lambda(epoch):
    warmup_epochs = 5      # Number of warmup epochs
    total_epochs = num_epochs  # Total number of epochs

    if epoch < warmup_epochs:
        # Linear warmup: increases from 0.2 to 1.0 over warmup_epochs
        return float(epoch + 1) / float(warmup_epochs)
    else:
        # Cosine decay: decreases from ~0.99 to 0.01 over the remaining epochs
        cosine_decay = 0.5 * (1 + math.cos(math.pi * (epoch - warmup_epochs + 1) / float(total_epochs - warmup_epochs)))
        return cosine_decay * 0.99 + 0.01  # Ensures lambda >= 0.01

In [ ]:
# Set hyperparameters
num_epochs = 30
batch_size = 64  # Set batch size to 64
learning_rate = 1e-3
weight_decay = 1e-4  # Slightly increased for better regularization
patience = 5  # Increased patience to allow more epochs before s.  topping

# Number of blocks to unfreeze
num_unfreeze_blocks = 4

# Move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Update the classification head and unfreeze n layers
num_classes = 10
unfreeze_last_n_blocks(model, num_unfreeze_blocks=num_unfreeze_blocks, num_classes=num_classes)

# Loss function, optimizer, and scheduler
criterion = nn.CrossEntropyLoss()
# criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
# criterion = FocalLoss(alpha=class_weights_tensor, gamma=2, reduction='mean')
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# optimizer = optim.SGD(model.parameters(),
#                       lr=learning_rate,          # Higher initial LR
#                       momentum=0.9,
#                       weight_decay=weight_decay)

# Choose scheduler: CosineAnnealingLR or ReduceLROnPlateau
# Option 1: CosineAnnealingLR
# scheduler = CosineAnnealingLR(
#     optimizer,
#     T_max=num_epochs,    # Number of epochs for a full cycle
#     eta_min=1e-6         # Minimum learning rate
# )

# Option 2: ReduceLROnPlateau (Commented out)
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,
    verbose=True,
    min_lr=1e-6
)

# Initialize the LambdaLR scheduler
# scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)


# Early Stopping Initialization
early_stopper = EarlyStopping(patience=patience, monitor="val_loss", mode="min")

# Initialize GradScaler for AMP
scaler = GradScaler()


# Unfreeze the last N blocks
unfreeze_last_n_blocks(model, num_unfreeze_blocks)

# Training loop
best_val_accuracy = 0.0
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Training Phase
    for images, labels in tqdm(train_loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        # Mixed Precision Forward Pass
        try:
            with torch.amp.autocast(device_type='cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
        except TypeError:
            # If device_type is not supported, fallback to default autocast
            with torch.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

        # Backward Pass with Scaler
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    # Training Metrics
    train_loss = running_loss / len(train_loader)
    train_accuracy = 100. * correct / total
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Training Loss: {train_loss:.4f}, Accuracy: {train_accuracy:.2f}%, LR: {current_lr:.6f}")

    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for val_images, val_labels in tqdm(val_loader, desc="Validation", leave=False):
            val_images, val_labels = val_images.to(device), val_labels.to(device)

            # Mixed Precision Forward Pass
            try:
                with torch.amp.autocast(device_type='cuda'):
                    val_outputs = model(val_images)
                    loss = criterion(val_outputs, val_labels)
            except TypeError:
                # If device_type is not supported, fallback to default autocast
                with torch.amp.autocast():
                    val_outputs = model(val_images)
                    loss = criterion(val_outputs, val_labels)

            val_loss += loss.item()

            _, val_predicted = val_outputs.max(1)
            val_total += val_labels.size(0)
            val_correct += val_predicted.eq(val_labels).sum().item()

    # Validation Metrics
    val_loss = val_loss / len(val_loader)
    val_accuracy = 100. * val_correct / val_total
    print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_accuracy:.2f}%")

    # Save the best model based on validation accuracy
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save(model.state_dict(), "best_eurosat_model.pth")
        print("Saved Best Model!")

    # Early Stopping Check
    early_stopper(val_loss)
    if early_stopper.early_stop:
        print("Early stopping triggered. Training stopped.")
        break

    # Step the scheduler
    scheduler.step(val_loss)

print("\nTraining Complete.")
print(f"Best Validation Accuracy: {best_val_accuracy:.2f}%")

<ipython-input-19-2d145a079106>:56: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Unfroze the last 4 blocks and updated the classification head to 10 classes.
Unfroze the last 4 blocks and updated the classification head to 10 classes.

Epoch 1/30


Training Loss: 0.7673, Accuracy: 73.49%, LR: 0.001000


Validation Loss: 0.5245, Accuracy: 81.73%
Saved Best Model!

Epoch 2/30


Training Loss: 0.5339, Accuracy: 81.56%, LR: 0.001000


...

Validation Loss: 0.3004, Accuracy: 90.52%

Epoch 22/30


Training Loss: 0.1437, Accuracy: 94.87%, LR: 0.000250


Validation Loss: 0.3104, Accuracy: 90.42%

Epoch 23/30


Training Loss: 0.1281, Accuracy: 95.38%, LR: 0.000250


Validation Loss: 0.3233, Accuracy: 90.49%
Early stopping triggered. Training stopped.

Training Complete.
Best Validation Accuracy: 90.62%


In [ ]:
# Test evaluation
model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for test_images, test_labels in test_loader:
        test_images, test_labels = test_images.to(device), test_labels.to(device)

        test_outputs = model(test_images)
        _, test_predicted = test_outputs.max(1)
        test_total += test_labels.size(0)
        test_correct += test_predicted.eq(test_labels).sum().item()

test_accuracy = 100. * test_correct / test_total
print(f"Test Accuracy: {test_accuracy:.2f}%")


Test Accuracy: 90.89%


### Visualization

In [ ]:
from sklearn.metrics import confusion_matrix
from tqdm import tqdm

def compute_and_print_confusion_matrix(model, dataloader, device, class_names, num_classes=10):
    all_preds = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Computing Confusion Matrix"):
            images = images.to(device)
            labels = labels.to(device)

            with torch.amp.autocast(device_type='cuda'):
                outputs = model(images)
                _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds, labels=range(num_classes))
    print("Confusion Matrix:")

    # Print header
    header = "\t" + "\t".join(class_names)
    print(header)

    # Print each row
    for i, row in enumerate(cm):
        row_str = class_names[i] + "\t" + "\t".join(map(str, row))
        print(row_str)

    return cm

class_names = [
    'Class0', 'Class1', 'Class2', 'Class3', 'Class4',
    'Class5', 'Class6', 'Class7', 'Class8', 'Class9'
]
# Assuming 'test_loader' is your DataLoader for the test set
confusion_matrix = compute_and_print_confusion_matrix(model, test_loader, device, class_names, num_classes=10)


Computing Confusion Matrix: 100%|██████████| 64/64 [00:11<00:00,  5.70it/s]

Confusion Matrix:
	Class0	Class1	Class2	Class3	Class4	Class5	Class6	Class7	Class8	Class9
Class0	401	0	5	8	0	9	12	1	1	0
Class1	0	442	5	2	0	1	0	0	0	0
Class2	6	1	412	4	1	10	7	3	2	0
Class3	12	3	7	261	18	17	17	27	18	0
Class4	0	0	0	13	344	0	0	9	6	0
Class5	5	4	2	11	0	285	10	0	1	0
Class6	21	1	0	12	4	6	337	7	2	0
Class7	1	0	1	19	16	0	16	380	2	0
Class8	0	2	3	9	5	4	0	0	348	5
Class9	0	0	0	0	0	0	0	0	7	439


Class 3 is under performing. Let's try class weighting to address this

In [ ]:
# Initialize an empty list to store all training labels
train_labels = []

# Disable gradient calculation for efficiency
with torch.no_grad():
    for batch in train_loader:
        # Assuming each batch is a tuple of (inputs, labels)
        _, labels = batch
        # Move labels to CPU and convert to NumPy array, then extend the list
        train_labels.extend(labels.cpu().numpy())

# Convert the list to a NumPy array for compatibility with sklearn
train_labels = np.array(train_labels)

print(f"Total training samples: {len(train_labels)}")

Total training samples: 18900


Computing Class Weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight


# Identify unique classes
classes = np.unique(train_labels)

# Compute class weights
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=classes,
                                     y=train_labels)

# Convert class weights to a PyTorch tensor
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Computed Class Weights:")
for cls, weight in zip(classes, class_weights):
    print(f"Class {cls}: Weight = {weight:.4f}")


Computed Class Weights:
Class 0: Weight = 0.8983
Class 1: Weight = 0.9009
Class 2: Weight = 0.9087
Class 3: Weight = 1.0757
Class 4: Weight = 1.0800
Class 5: Weight = 1.3656
Class 6: Weight = 1.0769
Class 7: Weight = 0.8861
Class 8: Weight = 1.0831
Class 9: Weight = 0.9026


Not much improvement after class weighting. Let's try Focal Loss

In [ ]:
# Initialize an empty list to store all training labels
train_labels = []

# Iterate through the train_loader to collect labels
for _, labels in train_loader:
    # Move labels to CPU and convert to NumPy array, then extend the list
    train_labels.extend(labels.cpu().numpy())

# Convert the list to a NumPy array for compatibility with sklearn
train_labels = np.array(train_labels)

print(f"Total training samples: {len(train_labels)}")

Total training samples: 18900


In [ ]:
# Convert class weights to a PyTorch tensor
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

# Move the tensor to the same device as your model (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = class_weights_tensor.to(device)

print("Class Weights Tensor:", class_weights_tensor)

Class Weights Tensor: tensor([0.8983, 0.9009, 0.9087, 1.0757, 1.0800, 1.3656, 1.0769, 0.8861, 1.0831,
        0.9026], device='cuda:0')


In [ ]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2, reduction='mean'):
        """
        :param alpha: Tensor of shape (num_classes,) representing class weights.
                      If None, all classes are treated equally.
        :param gamma: Focusing parameter for modulating factor (1-pt)^gamma.
        :param reduction: Specifies the reduction to apply to the output:
                          'none' | 'mean' | 'sum'
        """
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        """
        :param inputs: Predictions from the model (logits) of shape (batch_size, num_classes)
        :param targets: Ground truth labels of shape (batch_size,)
        :return: Focal loss value
        """
        if self.alpha is not None:
            if self.alpha.type() != inputs.data.type():
                self.alpha = self.alpha.type_as(inputs.data)
            alpha = self.alpha[targets]
        else:
            alpha = 1.0

        BCE_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)  # pt is the probability of the true class
        F_loss = alpha * (1 - pt) ** self.gamma * BCE_loss

        if self.reduction == 'mean':
            return F_loss.mean()
        elif self.reduction == 'sum':
            return F_loss.sum()
        else:
            return F_loss

In [ ]:
from collections import Counter
import numpy as np

# Assuming 'train_labels' is a NumPy array of all training labels
label_counts = Counter(train_labels)
print("Label Counts:", label_counts)


Label Counts: Counter({7: 2133, 0: 2104, 1: 2098, 9: 2094, 2: 2080, 3: 1757, 6: 1755, 4: 1750, 8: 1745, 5: 1384})


## Visualizing Attention Maps

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import cv2
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

In [ ]:
# Initialize a list to store attention maps
attention_maps = []

def capture_attention(module, input, output):
    """
    Hook function to capture attention weights from the Attention modules.
    Assumes that the Attention module returns attention weights or stores them internally.
    Modify this function based on your Attention module's implementation.
    """
    # Attempt to access attention weights
    try:
        # This assumes that the Attention module has an attribute 'attn_weights'
        # Adjust the attribute name based on your actual implementation
        attn = module.attn_weights  # Shape: (batch_size, num_heads, seq_len, seq_len)
    except AttributeError:
        # If 'attn_weights' attribute does not exist, try accessing output
        # This assumes that the Attention module returns attention weights as part of its output
        # Modify based on your implementation
        if isinstance(output, tuple):
            attn = output[1]  # Assuming the second element is attention weights
        else:
            raise AttributeError("Cannot access attention weights. Please check the Attention module's implementation.")

    # Append the attention weights to the list
    attention_maps.append(attn.detach().cpu())


In [ ]:
# Assuming 'model' is your BigEarthNetv2_0_ImageClassifier instance
for idx, block in enumerate(model.model.vision_encoder.blocks):
    # Register the forward hook on the 'attn' module of each block
    block.attn.register_forward_hook(capture_attention)
    print(f"Registered hook on Block {idx+1}'s Attention module.")


Registered hook on Block 1's Attention module.
Registered hook on Block 2's Attention module.
Registered hook on Block 3's Attention module.
Registered hook on Block 4's Attention module.
Registered hook on Block 5's Attention module.
Registered hook on Block 6's Attention module.
Registered hook on Block 7's Attention module.
Registered hook on Block 8's Attention module.
Registered hook on Block 9's Attention module.
Registered hook on Block 10's Attention module.
Registered hook on Block 11's Attention module.
Registered hook on Block 12's Attention module.


In [ ]:
def unnormalize(tensor, mean, std):
    """
    Unnormalize a tensor image with mean and standard deviation.
    """
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor


In [ ]:
def visualize_attention(image, attention_map, alpha=0.5, cmap='jet'):
    """
    Overlay the attention map on the original image.

    :param image: PIL Image (unnormalized)
    :param attention_map: 2D numpy array of attention values
    :param alpha: Transparency factor for the overlay
    :param cmap: Colormap for the attention map
    """
    # Convert PIL Image to numpy array
    image_np = np.array(image)

    # Normalize attention map
    attention_map = attention_map / attention_map.max()

    # Resize attention map to match image size
    attention_map_resized = cv2.resize(attention_map, (image_np.shape[1], image_np.shape[0]))
    attention_colormap = cv2.applyColorMap(np.uint8(255 * attention_map_resized), cv2.COLORMAP_JET)
    attention_colormap = cv2.cvtColor(attention_colormap, cv2.COLOR_BGR2RGB)

    # Overlay the attention map on the image
    overlayed_image = cv2.addWeighted(image_np, 1 - alpha, attention_colormap, alpha, 0)

    # Plot the overlayed image
    plt.figure(figsize=(8, 8))
    plt.imshow(overlayed_image)
    plt.axis('off')
    plt.title('Attention Map Overlay')
    plt.show()


In [ ]:
# Initialize a list to store attention maps
captured_attention_maps = []

def capture_attention(module, input, output):
    """
    Hook function to capture attention weights from Attention modules.

    :param module: The Attention module
    :param input: Input to the module
    :param output: Output from the module
    """
    # Attempt to access attention weights
    # Modify the attribute access based on your Attention module's implementation
    # For this example, we assume the Attention module stores attention weights in 'attn_weights'
    if hasattr(module, 'attn_weights'):
        attn = module.attn_weights  # Expected shape: (batch_size, num_heads, seq_len, seq_len)
        captured_attention_maps.append(attn.detach().cpu())
    else:
        # If attention weights are returned as part of the output
        if isinstance(output, tuple):
            attn = output[1]  # Assuming the second element is attention weights
            captured_attention_maps.append(attn.detach().cpu())
        else:
            print("Attention weights not found. Please verify the Attention module's implementation.")


In [ ]:
def get_single_image(test_loader, mean, std, selected_channels=[3, 2, 1]):
    """
    Retrieve a single image and its label from the test_loader.

    :param test_loader: DataLoader for the test dataset
    :param mean: List of mean values for each channel
    :param std: List of standard deviation values for each channel
    :param selected_channels: List of channel indices to select for RGB visualization
    :return: image_tensor, label, unnormalized_image
    """
    for images, labels in test_loader:
        image_tensor = images[0].unsqueeze(0).to(device)  # Shape: (1, C, H, W)
        label = labels[0].item()

        # Unnormalize for visualization
        unnormalized = unnormalize(images[0].clone(), mean, std)
        unnormalized = torch.clamp(unnormalized, 0, 1)  # Ensure values are within [0,1]

        # Select specific channels for RGB visualization
        # Example: Selecting Bands 4 (Red), 3 (Green), 2 (Blue) if applicable
        # Adjust 'selected_channels' based on your dataset's channel indexing
        selected_channels = [3, 2, 1]  # Zero-based indices
        unnormalized_rgb = unnormalized[selected_channels, :, :]  # Shape: (3, H, W)

        # Convert to PIL Image
        unnormalized_image = transforms.ToPILImage()(unnormalized_rgb.cpu())

        return image_tensor, label, unnormalized_image


In [ ]:
# --- Define Helper Functions ---

def unnormalize(tensor, mean, std):
    """
    Unnormalize a tensor image with mean and standard deviation.
    """
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor

def visualize_attention(image, attention_map, alpha=0.5, cmap='jet'):
    """
    Overlay the attention map on the original image.

    :param image: PIL Image (unnormalized)
    :param attention_map: 2D numpy array of attention values
    :param alpha: Transparency factor for the overlay
    :param cmap: Colormap for the attention map
    """
    # Convert PIL Image to numpy array
    image_np = np.array(image)

    # Normalize attention map
    attention_map = attention_map / attention_map.max()

    # Resize attention map to match image size
    attention_map_resized = cv2.resize(attention_map, (image_np.shape[1], image_np.shape[0]))

    # Apply a colormap (e.g., jet) to the attention map
    attention_colormap = cv2.applyColorMap(np.uint8(255 * attention_map_resized), cv2.COLORMAP_JET)
    attention_colormap = cv2.cvtColor(attention_colormap, cv2.COLOR_BGR2RGB)

    # Overlay the attention map on the image
    overlayed_image = cv2.addWeighted(image_np, 1 - alpha, attention_colormap, alpha, 0)

    # Plot the overlayed image
    plt.figure(figsize=(8, 8))
    plt.imshow(overlayed_image)
    plt.axis('off')
    plt.title('Attention Map Overlay')
    plt.show()

def visualize_attention_from_blocks(image, attention_maps, selected_blocks, patch_size=8, alpha=0.5, cmap='jet'):
    """
    Visualize attention maps from selected transformer blocks.

    :param image: PIL Image (unnormalized)
    :param attention_maps: List of attention tensors
    :param selected_blocks: List of block indices (0-based)
    :param patch_size: Size of each patch
    :param alpha: Transparency factor for the overlay
    :param cmap: Colormap for the attention map
    """
    for block_idx in selected_blocks:
        if block_idx >= len(attention_maps):
            print(f"Block index {block_idx} is out of range. Total blocks captured: {len(attention_maps)}")
            continue

        # Retrieve the attention map for the selected block
        attn = attention_maps[block_idx]  # Shape: (1, num_heads, seq_len, seq_len)

        # Average across all heads
        avg_attn = attn.mean(dim=1).squeeze(0).numpy()  # Shape: (seq_len, seq_len)

        # Remove [CLS] token attention (assuming the first token is [CLS])
        avg_attn = avg_attn[1:, 1:]  # Shape: (784, 784)

        # Reshape the attention map to (28, 28)
        try:
            attn_map = avg_attn.reshape(num_patches_per_dim, num_patches_per_dim)
        except:
            print(f"Cannot reshape attention map for Block {block_idx + 1}. Expected size ({num_patches_per_dim}, {num_patches_per_dim}), got {avg_attn.shape}")
            continue

        # Visualize the attention map overlay
        visualize_attention(image, attn_map, alpha=alpha, cmap=cmap)
        plt.title(f'Attention Map Overlay - Block {block_idx + 1}')
        plt.show()

# --- Register Forward Hooks ---

# Initialize a list to store attention maps
captured_attention_maps = []

def capture_attention(module, input, output):
    """
    Hook function to capture attention weights from Attention modules.

    :param module: The Attention module
    :param input: Input to the module
    :param output: Output from the module
    """
    # Attempt to access attention weights
    if hasattr(module, 'attn_weights'):
        attn = module.attn_weights  # Expected shape: (batch_size, num_heads, seq_len, seq_len)
        captured_attention_maps.append(attn.detach().cpu())
    else:
        # If attention weights are returned as part of the output
        if isinstance(output, tuple):
            attn = output[1]  # Assuming the second element is attention weights
            captured_attention_maps.append(attn.detach().cpu())
        else:
            print("Attention weights not found. Please verify the Attention module's implementation.")

# Register hooks on each Attention module within the transformer blocks
for idx, block in enumerate(model.model.vision_encoder.blocks):
    block.attn.register_forward_hook(capture_attention)
    print(f"Registered hook on Block {idx + 1}'s Attention module.")

# --- Select and Prepare an Input Image from test_loader ---

# Define the mean and std used during normalization (replace with your values if different)
mean = [0.485, 0.456, 0.406]  # Example values; adjust if different
std = [0.229, 0.224, 0.225]   # Example values; adjust if different

def get_single_image(test_loader, mean, std, selected_channels=[3, 2, 1]):
    """
    Retrieve a single image and its label from the test_loader.

    :param test_loader: DataLoader for the test dataset
    :param mean: List of mean values for each channel
    :param std: List of standard deviation values for each channel
    :param selected_channels: List of channel indices to select for RGB visualization
    :return: image_tensor, label, unnormalized_image
    """
    for images, labels in test_loader:
        image_tensor = images[0].unsqueeze(0).to(device)  # Shape: (1, C, H, W)
        label = labels[0].item()

        # Unnormalize for visualization
        unnormalized = unnormalize(images[0].clone(), mean, std)
        unnormalized = torch.clamp(unnormalized, 0, 1)  # Ensure values are within [0,1]

        # Select specific channels for RGB visualization
        # Example: Selecting Bands 4 (Red), 3 (Green), 2 (Blue) if applicable
        # Adjust 'selected_channels' based on your dataset's channel indexing
        selected_channels = [3, 2, 1]  # Zero-based indices
        unnormalized_rgb = unnormalized[selected_channels, :, :]  # Shape: (3, H, W)

        # Convert to PIL Image
        unnormalized_image = transforms.ToPILImage()(unnormalized_rgb.cpu())

        return image_tensor, label, unnormalized_image

# Retrieve a single image from test_loader
image_tensor, label, unnormalized_image = get_single_image(test_loader, mean, std)
print(f"Selected Image Label: {label}")

# --- Perform Forward Pass to Capture Attention Maps ---

# Clear any existing attention maps
captured_attention_maps.clear()

# Perform a forward pass
model.eval()  # Set model to evaluation mode

with torch.no_grad():
    output = model(image_tensor)

print(f"Captured {len(captured_attention_maps)} attention maps from transformer blocks.")

# --- Process and Visualize the Attention Maps ---

# Define patch size and image dimensions based on your model's configuration
patch_size = 8
image_size = 224
num_patches_per_dim = image_size // patch_size  # 28
total_patches = num_patches_per_dim ** 2       # 784

# Visualize attention from the last transformer block
if not captured_attention_maps:
    print("No attention maps were captured. Please verify the hook implementation.")
else:
    # Select the last attention map (Block 12, index 11)
    last_attention = captured_attention_maps[-1]  # Shape: (1, num_heads, seq_len, seq_len)

    # Average across all heads
    avg_attention = last_attention.mean(dim=1).squeeze(0).numpy()  # Shape: (seq_len, seq_len)

    # Remove [CLS] token attention (assuming the first token is [CLS])
    avg_attention = avg_attention[1:, 1:]  # Shape: (784, 784)

    # Reshape the attention map to (28, 28)
    try:
        attn_map = avg_attention.reshape(num_patches_per_dim, num_patches_per_dim)
    except:
        print(f"Cannot reshape attention map. Expected size ({num_patches_per_dim}, {num_patches_per_dim}), got {avg_attention.shape}")
        attn_map = None

    if attn_map is not None:
        # Visualize the attention map overlay
        visualize_attention(unnormalized_image, attn_map, alpha=0.5, cmap='jet')

    # Optional: Visualize attention from multiple blocks
    selected_blocks = [5, 8, 11]  # Blocks 6, 9, and 12 (0-based indexing)
    visualize_attention_from_blocks(unnormalized_image, captured_attention_maps, selected_blocks, patch_size=patch_size)


Registered hook on Block 1's Attention module.
Registered hook on Block 2's Attention module.
Registered hook on Block 3's Attention module.
Registered hook on Block 4's Attention module.
Registered hook on Block 5's Attention module.
Registered hook on Block 6's Attention module.
Registered hook on Block 7's Attention module.
Registered hook on Block 8's Attention module.
Registered hook on Block 9's Attention module.
Registered hook on Block 10's Attention module.
Registered hook on Block 11's Attention module.
Registered hook on Block 12's Attention module.
Selected Image Label: 9


AttributeError: Cannot access attention weights. Please check the Attention module's implementation.